In [4]:
import numpy as np
import scipy.stats as st
import scipy.optimize as opt

np.random.seed(42)

alpha = 0.05
k = 10
n = 100
m = np.array([5, 8, 6, 12, 14, 18, 11, 6, 13, 7])

### a) Проверка гипотезы о согласии с законом равномерного распределения

In [ ]:
nps_a = np.full(k, n * (1/10))
delt_pearson_a = np.sum((m - nps_a)**2 / nps_a)
pval_pearson_a = st.chi2(k - 1).sf(delt_pearson_a)

print(f"Критерий Пирсона: delta = {delt_pearson_a:.2f}, p-value = {pval_pearson_a:.4f}")

F_emp = np.cumsum(m) / n
F_th_a = np.cumsum(np.full(k, 1/10))

delt_kolm_a = np.sqrt(n) * np.max(np.abs(F_emp[:9] - F_th_a[:9]))
pval_kolm_a = st.kstwobign.sf(delt_kolm_a)

print(f"Критерий Колмогорова: delta = {delt_kolm_a:.3f}, p-value = {pval_kolm_a:.4f}")

**Сравнение результатов:**
Оба критерия пришли к одному выводу: нет веских оснований отвергать гипотезу о равномерном распределении (оба $p\text{-value} > 0.05$).

### b) Проверка гипотезы о согласии с законом нормального распределения

In [9]:
def get_p_norm(loc, scale):
    pi = st.norm.cdf(np.arange(1, 10), loc=loc, scale=scale)
    p = np.zeros(10)
    p[0] = pi[0]
    p[1:9] = np.diff(pi)
    p[9] = 1 - pi[-1]
    return p

def neg_log_lik(thetas):
    p = np.clip(get_p_norm(thetas[0], thetas[1]), 1e-10, 1.0)
    return -np.sum(m * np.log(p))

res = opt.differential_evolution(neg_log_lik, bounds=[(0, 10), (0.1, 10)], seed=42)
theta1, theta2 = res.x

p_norm = get_p_norm(theta1, theta2)
nps_b = n * p_norm
delt_pearson_b = np.sum((m - nps_b)**2 / nps_b)
pval_pearson_b = st.chi2(k - 1 - 2).sf(delt_pearson_b)

print(f"Оценки ОМПГ: theta1 = {theta1:.2f}, theta2 = {theta2**2:.2f}")
print(f"Критерий Пирсона: delta = {delt_pearson_b:.2f}, p-value = {pval_pearson_b:.4f}")

Оценки ОМПГ: theta1 = 5.29, theta2 = 7.18
Критерий Пирсона: delta = 9.80, p-value = 0.2000


In [10]:
from scipy.optimize import minimize

F_th_b = st.norm.cdf(np.arange(1, 10), loc=theta1, scale=theta2)
delt_kolm_b = np.sqrt(n) * np.max(np.abs(F_emp[:9] - F_th_b))

N_BS = 5000
delta_stars = []

def fast_mle(m_samp):
    def nll(thetas):
        p = np.clip(get_p_norm(thetas[0], thetas[1]), 1e-10, 1.0)
        return -np.sum(m_samp * np.log(p))
    res = minimize(nll, [theta1, theta2], bounds=[(0, 10), (0.1, 10)], method='L-BFGS-B')
    return res.x

for _ in range(N_BS):
    m_star = np.random.multinomial(n, p_norm)
    F_emp_star = np.cumsum(m_star) / n

    mu_star, std_star = fast_mle(m_star)

    F_th_star = st.norm.cdf(np.arange(1, 10), loc=mu_star, scale=std_star)

    d_star = np.sqrt(n) * np.max(np.abs(F_emp_star[:9] - F_th_star))
    delta_stars.append(d_star)

pval_kolm_boot = np.mean(np.array(delta_stars) >= delt_kolm_b)

print(f"Критерий Колмогорова: delta = {delt_kolm_b:.3f}, p-value = {pval_kolm_boot:.4f}")

Критерий Колмогорова: delta = 0.441, p-value = 0.3576


**Сравнение результатов:**
Оба критерия пришли к одному выводу: нет веских оснований отвергать гипотезу о равномерном распределении. В целом, нормальный закон описывает данную выборку заметно лучше, чем равномерное распределение из пункта **a**.
